In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from datetime import datetime

import multiprocessing
from multiprocessing import Pool
import yfinance as yf

start = '2010-01-01'
end = '2024-12-31'

tickers = ['VNQ', 'SPY', 'GLD', 'BTC-USD']

benchmark_weights = np.array([0.1, 0.8, 0.05, 0.05, 0])

df = yf.download(tickers, start=start, end=end, interval='1mo')
df = df['Close']
df = df.dropna().copy()

C:\Users\souza\AppData\Local\Temp\ipykernel_23976\2174190880.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, start=start, end=end, interval='1mo')
[*********************100%***********************]  4 of 4 completed


In [2]:
print(df.head())

Ticker         BTC-USD         GLD         SPY        VNQ
Date                                                     
2014-09-01  386.944000  116.209999  162.429459  45.781052
2014-10-01  338.321014  112.660004  167.031982  50.798351
2014-11-01  378.046997  112.110001  171.620636  51.814445
2014-12-01  320.192993  113.580002  170.245667  52.090961
2015-01-01  217.464005  123.449997  166.113205  56.418797


In [3]:
return_tickers = []
for ticker in tickers:
    rtick = f"{ticker}_return"
    df[rtick] = df[ticker].pct_change()
    return_tickers.append(rtick)

In [5]:
# split into train and test
n_train = int(0.8 * len(df))
df_train = df.iloc[:n_train]
df_test = df.iloc[n_train:]

In [6]:
# pre-compute returns
train_returns = df_train[return_tickers].dropna().to_numpy()
test_returns = df_test[return_tickers].dropna().to_numpy()


In [9]:
# add a column of 0s for cash
z = np.zeros((train_returns.shape[0], 1))
train_returns = np.hstack((train_returns, z))

In [10]:
train_returns

array([[ 1.09593368e-01,  2.83355239e-02, -3.05481064e-02,
        -1.25658973e-01,  0.00000000e+00],
       [ 2.00025037e-02,  2.74717063e-02, -4.88197261e-03,
         1.17420973e-01,  0.00000000e+00],
       [ 5.33665772e-03, -8.01167922e-03,  1.31121328e-02,
        -1.53033894e-01,  0.00000000e+00],
       [ 8.30822654e-02, -2.42735197e-02,  8.68990576e-02,
        -3.20834593e-01,  0.00000000e+00],
       [-3.67416192e-02,  5.62044644e-02, -5.90521949e-02,
         1.69218791e-01,  0.00000000e+00],
       [ 1.12751629e-02, -2.00796486e-02, -2.15220379e-02,
        -3.94827460e-02,  0.00000000e+00],
       [-5.28477775e-02,  1.43414910e-02, -1.67167372e-03,
        -3.30802656e-02,  0.00000000e+00],
       [-3.02366424e-03,  1.28561903e-02,  5.55210405e-03,
        -2.52175643e-02,  0.00000000e+00],
       [-5.62294575e-02, -2.50544623e-02, -1.51621012e-02,
         1.42847162e-01,  0.00000000e+00],
       [ 6.84003578e-02,  2.75632175e-02, -6.62098626e-02,
         8.20231863e-02

In [12]:
def evolution_strategy(
    f,
    population_size,
    sigma,
    lr,
    initial_params,
    num_iters,
    pool):

    # assume initial params is a 1-D array
    num_params = len(initial_params)
    reward_per_iteration = np.zeros(num_iters)

    params = initial_params
    for t in range(num_iters):
        t0 = datetime.now()
        N = np.random.randn(population_size, num_params)

        ## fast way
        R = pool.map(f, [params + sigma * N[j] for j in range(population_size)])
        R = np.array(R)

        m = R.mean()
        s = R.std()
        if s == 0:
            # we can't apply the following equation
            print("Skipping")
            continue

        A = (R - m) /s

        reward_per_iteration[t] = m
        params = params + lr / (population_size * sigma) * np.dot(N.T, A)

        print("Iter:", t, "Avg Reward: %.3f" % m, "Max: %.3f" % R.max(), "Duration:", datetime.now() - t0)

        return params, reward_per_iteration
        

In [13]:
def softmax(a):
    c = np.max(a, axis=1, keepdims=True)
    exp_a = np.exp(a - c)
    sum_exp_a = np.sum(exp_a, axis=1, keepdims=True)
    return exp_a / sum_exp_a

In [14]:
def sortino_ration(returns, target=0):
    downside = returns[returns < target]
    downside_std = np.sqrt(np.mean((downside - target)**2)) \
        if len(downside) > 0 else 1e-8
    return (np.mean(returns) - target) / downside_deviation
    